# 3. Select interaction and propagation schemes

Keep the specimen fixed while changing one model choice at a time. Scalar, Jones and Stokes select field representations; multislice on/off changes within-sample diffraction; Fraunhofer/RS changes only exit-to-camera propagation.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/scattering_calculator').is_dir())
PAPER = ROOT / 'paper/scattering_calculator'
sys.path.insert(0, str(PAPER))
import experiments as ex
import workflow as wf

def show(name):
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(plt.imread(OUT / (name + '.png')))
    ax.axis('off')
    plt.show()


In [ ]:
# EDIT HERE: sample dimensions are nm; detector lengths are m.
sample = ex.FTHConfig(
    energy_eV=778., n=192, dx_nm=10.,
    layers_nm=(('Au',300.), ('SiN',20.), ('Pt',2.), ('Co',10.), ('Pt',2.)),
    max_slice_nm=5., object_radius_nm=180., reference_radius_nm=30.,
    reference_xy_nm=(600.,0.), beam_sigma_nm=650.,
    domain_period_nm=110., propagate=True,
    energies_eV=(772.,775.,778.,780.,783.,790.,795.,800.),
)
camera = wf.DetectorGeometry(shape=(128,128), pixel_size_m=13.5e-6,
                             distance_m=.05, method='fraunhofer')
OUT = PAPER / 'results' / 'notebook_model_options'
OUT.mkdir(parents=True, exist_ok=True)


## Scalar, Jones and pure-state Stokes
The default circular input and zero linear material channel make Scalar a useful eigenmode control. Stokes shares its Jones carrier; agreement checks consistency.

In [ ]:
cases = {mode: ex.fth_case(config=sample, mode=mode)
         for mode in ('Scalar','Jones','Stokes')}
ref = cases['Jones']['difference']
limit = abs(ref).max()
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for ax, (mode, case) in zip(axes, cases.items()):
    error = np.linalg.norm(case['difference']-ref)/np.linalg.norm(ref)
    ax.imshow(case['difference'], origin='lower', cmap='RdBu_r', vmin=-limit, vmax=limit)
    ax.set_title(f'{mode}: relative L2 {error:.3g}')
plt.show()


## Multislice versus a projection control
`propagate=False` retains local interactions and omits transverse diffraction between slices. Compare on the same q grid and scale.

In [ ]:
projection = ex.fth_case(config=replace(sample, propagate=False))
print('Projection / multislice relative difference:',
      np.linalg.norm(projection['difference']-ref)/np.linalg.norm(ref))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
for ax, image, title in zip(axes, (ref,projection['difference']), ('Multislice','Projection control')):
    ax.imshow(image, origin='lower', cmap='RdBu_r', vmin=-limit, vmax=limit)
    ax.set_title(title)
plt.show()


## Switch exit-to-detector propagation
A small detector makes direct RS integration practical. Both calculations receive the same complex exit fields. These panels use separate relative scales: raw brightness is not a calibrated backend comparison. The legacy FFT and RS transverse-coordinate conventions differ; match conventions, source support, sampling and normalization before computing physical error metrics.

In [ ]:
from matplotlib.colors import LogNorm
small = replace(camera, shape=(24,24))
ff, _ = wf.project(cases['Jones'], small)
rs, _ = wf.project(cases['Jones'], replace(small, method='rayleigh_sommerfeld'))
fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
for ax, image, title in zip(axes, (ff[0],rs[0]), ('Fraunhofer','Rayleigh–Sommerfeld')):
    ax.imshow(image/image.max(), origin='lower', norm=LogNorm(1e-6,1))
    ax.set_title(title + ' (own peak = 1)')
    ax.set_xlabel('detector column')
plt.show()
np.savez_compressed(OUT / 'propagation_options.npz', fraunhofer=ff, rayleigh_sommerfeld=rs)
